# Notebook 01: Raw to Processed
## Amazon Fine Food Reviews: Data Cleaning, Validation & Anonymisation
This notebook reads the raw CSV from the `/raw` zone, applies schema validation, 
quarantines invalid records, anonymises PII fields, and writes cleaned Parquet 
output to the `/processed` zone partitioned by year.

Helper functions used in this notebook are defined in `modules/data_helpers.py`.
The logic here follows the same structure as those functions:
- Validation 
- Quarantine 
- Anonymisation
- Feature engineering 

These have been implemented inline for Databricks compatibility.

In [0]:
# Configuration and storage connection
STORAGE_ACCOUNT = "stcw2bdcbh"
STORAGE_KEY = dbutils.secrets.get(scope="kv-scope", key="storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

RAW_PATH = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/amazon-reviews/Reviews.csv"
PROCESSED_PATH = f"abfss://processed@{STORAGE_ACCOUNT}.dfs.core.windows.net/amazon-reviews/"
QUARANTINE_PATH = f"abfss://quarantine@{STORAGE_ACCOUNT}.dfs.core.windows.net/rule=schema_violation/"

print("Configuration set.")
print(f"Raw path: {RAW_PATH}")
print(f"Processed path: {PROCESSED_PATH}")

Configuration set.
Raw path: abfss://raw@stcw2bdcbh.dfs.core.windows.net/amazon-reviews/Reviews.csv
Processed path: abfss://processed@stcw2bdcbh.dfs.core.windows.net/amazon-reviews/


### Step 1: Load Raw Data
Read the CSV with enforced schema. Any rows that cannot be cast to the expected 
types will produce nulls. This will be caught in the next step.

In [0]:
# Load raw data and validate schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define expected schema
expected_schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("ProductId", StringType(), True),
    StructField("UserId", StringType(), True),
    StructField("ProfileName", StringType(), True),
    StructField("HelpfulnessNumerator", IntegerType(), True),
    StructField("HelpfulnessDenominator", IntegerType(), True),
    StructField("Score", IntegerType(), True),
    StructField("Time", IntegerType(), True),
    StructField("Summary", StringType(), True),
    StructField("Text", StringType(), True)
])

# Load raw CSV
df_raw = spark.read.csv(
    RAW_PATH,
    header=True,
    schema=expected_schema,
    multiLine=True,
    escape='"'
)

raw_count = df_raw.count()
print(f"Raw row count: {raw_count}")
print(f"Columns: {df_raw.columns}")
df_raw.printSchema()

Raw row count: 568454
Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']
root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Time: integer (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)



### Step 2: Data Quality Checks & Quarantine
Null counts are logged for all columns. Rows missing a `Score` or `Id` are 
written to the `/quarantine` zone and excluded from downstream processing.

In [0]:
# Data quality checks and quarantine
from pyspark.sql.functions import col

# Check for nulls in critical columns
null_counts = df_raw.select([
    col(c).isNull().cast("int").alias(c) 
    for c in df_raw.columns
]).groupBy().sum().collect()[0].asDict()

print("Null counts per column:")
for col_name, count in null_counts.items():
    print(f"  {col_name}: {count}")

# Quarantine rows with null Score (critical for analysis)
df_quarantine = df_raw.filter(col("Score").isNull() | col("Id").isNull())
quarantine_count = df_quarantine.count()
print(f"\nRows sent to quarantine: {quarantine_count}")

# Write quarantine records
if quarantine_count > 0:
    df_quarantine.write.mode("overwrite").parquet(QUARANTINE_PATH)
    print(f"Quarantine written to: {QUARANTINE_PATH}")

# Valid records only
df_valid = df_raw.filter(col("Score").isNotNull() & col("Id").isNotNull())
valid_count = df_valid.count()
print(f"Valid rows after quarantine: {valid_count}")

# Fail-fast if more than 50% rows lost
if valid_count < raw_count * 0.5:
    raise Exception(f"FAIL: Too many rows lost. Raw: {raw_count}, Valid: {valid_count}")

print("Data quality check passed.")

Null counts per column:
  sum(Id): 0
  sum(ProductId): 0
  sum(UserId): 0
  sum(ProfileName): 0
  sum(HelpfulnessNumerator): 0
  sum(HelpfulnessDenominator): 0
  sum(Score): 0
  sum(Time): 0
  sum(Summary): 0
  sum(Text): 0

Rows sent to quarantine: 0
Valid rows after quarantine: 568454
Data quality check passed.


### Step 3: Deduplication
Duplicate rows are removed based on `Id`. Row counts before and after are logged.

In [0]:
# Deduplication
df_deduped = df_valid.dropDuplicates(["Id"])

deduped_count = df_deduped.count()
duplicates_removed = valid_count - deduped_count
print(f"Rows before dedup: {valid_count}")
print(f"Rows after dedup: {deduped_count}")
print(f"Duplicates removed: {duplicates_removed}")

Rows before dedup: 568454
Rows after dedup: 568454
Duplicates removed: 0


### Step 4: Anonymisation
`UserId` is SHA-256 hashed with a salt retrieved from Azure Key Vault and **never** 
hardcoded. `ProfileName` is fully redacted as it is a direct identifier.

In [0]:
# Anonymisation
# SHA-256 hash UserId with salt from Key Vault
# ProfileName masked entirely (too identifying)

from pyspark.sql.functions import sha2, concat, lit

SALT = dbutils.secrets.get(scope="kv-scope", key="anonymisation-salt")

df_anonymised = df_deduped \
    .withColumn("UserId", sha2(concat(col("UserId"), lit(SALT)), 256)) \
    .withColumn("ProfileName", lit("REDACTED"))

print("Anonymisation applied:")
print("  - UserId: SHA-256 hashed with salt")
print("  - ProfileName: redacted")

df_anonymised.select("Id", "UserId", "ProfileName").show(3, truncate=False)

Anonymisation applied:
  - UserId: SHA-256 hashed with salt
  - ProfileName: redacted
+---+----------------------------------------------------------------+-----------+
|Id |UserId                                                          |ProfileName|
+---+----------------------------------------------------------------+-----------+
|1  |aec2e522b9a0610123e507e153316a87c3b02961b2dfde18a8c48ac57a434e82|REDACTED   |
|6  |2f1a8e45dfd141fb9a49c73b5bbed9c664739921d08b433b231bbbc9e746d4e7|REDACTED   |
|12 |1fb99a7395b69503b6423a8c03c7cb978746a00a3f547dcb2f65cf4ba7ca804e|REDACTED   |
+---+----------------------------------------------------------------+-----------+
only showing top 3 rows



### Step 5: Feature Engineering
Unix timestamps are converted to readable dates. A helpfulness ratio and 
sentiment band are derived for use in the analytical tables.

In [0]:
# Type casting and feature engineering
from pyspark.sql.functions import from_unixtime, year, month, when

df_processed = df_anonymised \
    .withColumn("ReviewDate", from_unixtime(col("Time")).cast("date")) \
    .withColumn("ReviewYear", year(from_unixtime(col("Time")))) \
    .withColumn("ReviewMonth", month(from_unixtime(col("Time")))) \
    .withColumn("HelpfulnessRatio", 
        when(col("HelpfulnessDenominator") == 0, None)
        .otherwise(col("HelpfulnessNumerator") / col("HelpfulnessDenominator"))
    ) \
    .withColumn("SentimentBand",
        when(col("Score") >= 4, "Positive")
        .when(col("Score") == 3, "Neutral")
        .otherwise("Negative")
    ) \
    .drop("Time")

print(f"Processed schema:")
df_processed.printSchema()
df_processed.show(3)

Processed schema:
root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = false)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)
 |-- ReviewDate: date (nullable = true)
 |-- ReviewYear: integer (nullable = true)
 |-- ReviewMonth: integer (nullable = true)
 |-- HelpfulnessRatio: double (nullable = true)
 |-- SentimentBand: string (nullable = false)

+---+----------+--------------------+-----------+--------------------+----------------------+-----+--------------------+--------------------+----------+----------+-----------+----------------+-------------+
| Id| ProductId|              UserId|ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|             Summary|                Text|ReviewDate|ReviewYear|Review

### Step 6: Write to Processed Zone
Output is written as Parquet partitioned by `ReviewYear`. Parquet was chosen 
over CSV for columnar compression. It also enables faster downstream reads in Spark.

In [0]:
# Write to processed zone as Parquet
df_processed.write \
    .mode("overwrite") \
    .partitionBy("ReviewYear") \
    .parquet(PROCESSED_PATH)

processed_count = df_processed.count()
print(f"Written to processed zone: {processed_count} rows")
print(f"Path: {PROCESSED_PATH}")
print(f"Format: Parquet, partitioned by ReviewYear")
print("Notebook 01 complete.")

Written to processed zone: 568454 rows
Path: abfss://processed@stcw2bdcbh.dfs.core.windows.net/amazon-reviews/
Format: Parquet, partitioned by ReviewYear
Notebook 01 complete.
